# Streets Network - clean, fully attributed build

Rebuilds every attribute on the PROTECT street network in one pass and writes a **new** feature
class, `Streets_Network_Tahoe_Clean`. The published `Streets_Network_Tahoe` is read, never modified.

It consolidates the work that was spread across three notebooks, each of which wrote fields into the
layer in place:

| Group | Origin | Fields |
|---|---|---|
| Hazard exposure | `Hazard_Vulnerability.ipynb` | flood, 100-yr flood, high-severity fire, landslide |
| Equity | `Asset_Criticality.ipynb` + `Asset_Scoring.ipynb` sec. 5 | block-group maxima, normalized inputs, Equity score |
| Traffic | `Asset_Criticality.ipynb` (AADT join) + `Asset_Scoring.ipynb` sec. 6 | StreetLight AADT, filled AADT, Jenks score |
| Detour | `scripts/ra2ce_v2` SLR update table + `Asset_Criticality.ipynb` (neighbor fill) + `Asset_Scoring.ipynb` sec. 6 | single-link-redundancy detour length, Jenks score |
| OD criticality | `scripts/ra2ce_v2/od_stochastic_assignment.ipynb` update table (`OD_METHOD`) | 20 traffic / routed / par_alt fields + 3 QA fields |
| Criticality | `html/criticality-index.html` method + OD traffic; weights traffic 0.3 / detour 0.3 / OD 0.3 / equity 0.1 | sub-indices + composite score |

## Fixes applied (vs. the published layer)

1. **OD criticality on the corrected network, routed by travel time with stochastic assignment.**
   Replaces the `scripts/ra2ce` values. The corrected `ra2ce_v2` network recovers the 388 streets that read
   NULL because of the RA2CE parallel-link collapse. Routing comes from
   `scripts/ra2ce_v2/od_stochastic_assignment.ipynb`: travel time from speeds by road class, averaged over
   100 noisy draws, so gridded neighbourhoods load their collectors instead of one arbitrary staircase of
   residential blocks. Set `OD_METHOD = "ra2ce"` to use RA2CE's shortest-distance routing instead.
2. **SLR rerun on the corrected network.** The published detour values came from the July
   `scripts/ra2ce` run (`tahoe_slr`), on the graph with collapsed parallel links and straight-line
   edge lengths, joined to streets by a one-sided buffer overlap. They now come from
   `ra2ce_v2` (`pipeline.py --stages slr`, then `export_slr.py`): SLR on the split + repaired graph,
   mapped to the true segment geometry through RA2CE's `rfid_c` mapping, then joined with the same
   two-sided rule as the OD fields. Every street now has a value (6,117 were NULL). Self-loops (the
   "alternative" from a node to itself is 0 m) and negative detours are set to 0 in the export.
3. **SLR neighbor fill consolidated.** The published layer split it across two fields: the detour = 0
   fill lived only in `slr_diff_length_max_from_neighbors`, while the NULL fill was written into
   `slr_diff_length_max` itself, so the Jenks score saw one fill and not the other. Now one fill, into
   `slr_diff_length_max`, before the Jenks score. On the corrected network it applies to about 50
   major-road streets instead of ~1,450; most "no detour" major roads were collapse artifacts.
4. **Major-road AADT gaps filled from neighbors.** Trunk / primary / secondary / tertiary segments with
   no StreetLight match previously got 0 (or stayed NULL: 48 trunk segments), so they scored as the
   lowest-traffic class. They now take the max AADT of touching segments. `aadt_source` flags them.
5. **Equity gaps filled from the nearest block group.** Streets that touch no block group polygon
   (32 in the published layer) take the nearest one. `equity_bg_match` flags them.
6. **Broken fields dropped.** `diff_length_max` / `detour_max` came from a buffered-line join that
   referenced undefined variables and hold a single constant; empty editor-tracking, `surface` and
   `speed_limit_kmh` fields are dropped.
7. **`in_flood_zone_100` added.** It was computed in `Hazard_Vulnerability.ipynb` but never reached
   the Tahoe layer.
8. **Landslide statistics without a scratch GDB.** Sampled along each line with rasterio (one sample per
   raster cell touched) instead of `ZonalStatisticsAsTable`, so there is no schema lock or scratch copy.
   Results match the published values exactly on most streets; differences sit mainly on segments
   under 100 m, where the two methods pick up different cells.
9. **Deterministic AADT join.** A street lying fully inside several overlapping StreetLight buffers
   (both directions of a road) ties on overlap, and the old loop kept whichever candidate the spatial
   index returned first. Ties now go to the largest AADT, the conservative choice for criticality.

**Kernel:** `arcgispro-py3-plotly` (needs geopandas, shapely, rasterio and jenkspy; `arcpy` is only
used for the final FGDB write). Close the output feature class in ArcGIS Pro before re-running with
`OVERWRITE = True`.

In [1]:
import time
import warnings
from pathlib import Path

import jenkspy
import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import rasterio
import shapely
from rasterio.transform import rowcol
from rasterio.windows import from_bounds
from shapely.strtree import STRtree

warnings.filterwarnings("ignore", message=r"Measured \(M\) geometry types are not supported")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)

## 1. Configuration

In [2]:
ANALYSIS_DIR = Path(r"F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis")
GDB = ANALYSIS_DIR / "PROTECT_analysis.gdb"

BASE_LAYER        = "Streets_Network_Tahoe"     # geometry + identity fields (read only)
FLOOD_LAYER       = "FEMA_Flood_Zone"
FIRE_LAYER        = "ForestManagementZone_Identity_HighSeverityProbability"
BLOCK_GROUP_LAYER = "Block_Group_Layer"
AADT_LAYER        = "AADT_Streetlight_Attributed"
LANDSLIDE_RASTER  = ANALYSIS_DIR / "usgs_landslide_risk.tif"

# Repo paths resolve from the notebook folder (scripts/) or the repo root
REPO = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
V2_OUT = REPO / "scripts" / "ra2ce_v2" / "output"
OD_UPDATE_DIR = V2_OUT / "protect_va_update"

# OD criticality source:
#   "stochastic" - od_stochastic_assignment.ipynb: travel time by road class + stochastic assignment
#   "ra2ce"      - RA2CE shortest-distance routing (pipeline.py --stages route)
OD_METHOD = "stochastic"
OD_UPDATE_CSV = {
    "stochastic": V2_OUT / "od_stochastic" / "protect_va_update" / "protect_va_od_update_stochastic.csv",
    "ra2ce":      OD_UPDATE_DIR / "protect_va_od_update.csv",
}[OD_METHOD]
SLR_UPDATE_CSV = OD_UPDATE_DIR / "protect_va_slr_update.csv"  # ra2ce_v2 export_slr.py
OD_PUBLISHED  = OD_UPDATE_DIR / "protect_va_published.gpkg"   # key check only, optional

OUT_DIR  = REPO / "scripts" / "outputs" / "streets_network_clean"   # gitignored
OUT_GPKG = OUT_DIR / "streets_network_tahoe_clean.gpkg"
OUT_FC   = "Streets_Network_Tahoe_Clean"
WRITE_FGDB = True
OVERWRITE  = False    # True replaces an existing OUT_FC

MAJOR_CLASSES   = ["trunk", "primary", "secondary", "tertiary"]
EQUITY_FIELDS   = ["Below_Poverty_Household", "Speak_English_Not_Well_Not_at_A",
                   "Vehicle_Available_0", "With_Disability"]
LINE_JOIN_BUFFER_M  = 5     # Asset_Criticality.assign_max_line_join_values defaults
LINE_JOIN_MIN_SHARE = 0.5
TIE_TOL_M = 0.5             # overlaps this close to the best count as a tie
NEIGHBOR_PASSES = 5         # lets a fill walk along a chain of unmatched segments
JENKS_CLASSES   = 10

# Criticality score weights (must sum to 1). "od" is the mean of the four OD traffic fields.
CRIT_WEIGHTS = {"traffic": 0.3, "detour": 0.3, "od": 0.3, "equity": 0.1}

OUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Base network

Only the identity fields are read from `Streets_Network_Tahoe`; every analytic field is rebuilt below.
The layer's OBJECTID is kept as `src_objectid`: it is the key for the `ra2ce_v2` update table and for
PROTECT_VA (`segment_id` is not unique, so it cannot be the key).

The full published attribute table is read separately as `published`, for the before/after QA in section 10.

In [3]:
BASE_FIELDS = ["segment_id", "from_connector", "to_connector", "class", "name",
               "length_m", "DriveTime", "In_Basin"]

t0 = time.time()
streets = pyogrio.read_dataframe(GDB, layer=BASE_LAYER, columns=BASE_FIELDS, fid_as_index=True)
streets = streets.rename_axis("src_objectid").reset_index().sort_values("src_objectid", ignore_index=True)
streets["geometry"] = shapely.force_2d(streets.geometry.values)
streets = gpd.GeoDataFrame(streets, geometry="geometry", crs=streets.crs)

published = (pyogrio.read_dataframe(GDB, layer=BASE_LAYER, read_geometry=False, fid_as_index=True)
             .rename_axis("src_objectid").sort_index())

assert streets.src_objectid.is_unique
assert streets.geometry.notna().all() and not streets.geometry.is_empty.any()
print(f"{len(streets):,} streets, CRS {streets.crs.to_epsg()}, read in {time.time() - t0:.0f}s")
print(streets["class"].value_counts().to_dict())

18,285 streets, CRS 26910, read in 13s
{'service': 8566, 'residential': 6306, 'primary': 928, 'tertiary': 836, 'unclassified': 749, 'trunk': 442, 'secondary': 301, 'unknown': 146, 'living_street': 11}


### Shared helpers

* `touching_pairs` - every (segment, touching segment) pair, the vectorized equivalent of the
  `geom.touches()` loop in `assign_max_neighboring_line_values`.
* `neighbor_max_fill` - max value of touching donors, repeated so a fill can walk along a chain of
  unmatched segments. Fill targets are never donors, so a suspect value cannot spread.
* `best_overlap_join` - port of `assign_max_line_join_values`: buffer the source lines, and give each
  street the source line whose buffer covers the most of it, provided that is at least
  `min_share` of the street's length.
* `jenks_score` - port of `classify_with_jenks`: score 1..n on Jenks breaks, nulls score 1.

In [4]:
GEOMS = streets.geometry.values
_tree = STRtree(GEOMS)
_i, _j = _tree.query(GEOMS, predicate="touches")
TOUCH_PAIRS = (_i[_i != _j], _j[_i != _j])
print(f"{len(TOUCH_PAIRS[0]):,} touching pairs")


def neighbor_max_fill(values, target, pairs=TOUCH_PAIRS, passes=NEIGHBOR_PASSES):
    """Fill target rows with the max of touching non-target values.

    Returns (filled_values, n_filled). Rows that are not targets come back unchanged;
    targets with no reachable donor come back NaN.
    """
    vals = np.asarray(values, dtype=float).copy()
    target = np.asarray(target, dtype=bool)
    vals[target] = np.nan
    donor = ~np.isnan(vals)
    pending = target.copy()
    recv, give = pairs
    for _ in range(passes):
        m = pending[recv] & donor[give]
        if not m.any():
            break
        best = pd.Series(vals[give[m]]).groupby(recv[m]).max()
        idx = best.index.to_numpy()
        vals[idx] = best.to_numpy()
        donor[idx] = True
        pending[idx] = False
    return vals, int((target & ~pending).sum())


def best_overlap_join(target, source, fields, buffer_m=LINE_JOIN_BUFFER_M,
                      min_share=LINE_JOIN_MIN_SHARE, tie_tol_m=TIE_TOL_M):
    """For each target line, the source line whose buffer covers the most of it (>= min_share).

    Candidates within tie_tol_m of the best overlap count as tied (common where RA2CE / StreetLight
    lines overlap, e.g. both directions of a road); a tie goes to the largest value of fields[0].
    """
    src = source.to_crs(target.crs).reset_index(drop=True)
    src = src[src.geometry.notna() & ~src.geometry.is_empty].reset_index(drop=True)
    bufs = shapely.buffer(src.geometry.values, buffer_m)
    tgeom = target.geometry.values
    ti, si = STRtree(bufs).query(tgeom)
    overlap = shapely.length(shapely.intersection(tgeom[ti], bufs[si]))
    keep = (overlap > 0) & (overlap >= min_share * shapely.length(tgeom[ti]))
    cand = pd.DataFrame({"t": ti[keep], "s": si[keep], "ov": overlap[keep]})
    cand["v"] = pd.to_numeric(src[fields[0]], errors="coerce").to_numpy()[cand.s]
    cand = cand[cand.ov >= cand.groupby("t").ov.transform("max") - tie_tol_m]
    n_tied = int((cand.groupby("t").size() > 1).sum())
    best = (cand.sort_values(["t", "v", "ov", "s"], ascending=[True, False, False, True],
                             na_position="last")
            .drop_duplicates("t"))
    out = pd.DataFrame(index=target.index, columns=fields, dtype=object)
    out.iloc[best.t.to_numpy(), :] = src.loc[best.s.to_numpy(), fields].to_numpy()
    print(f"  {len(best):,} of {len(target):,} streets matched "
          f"(buffer {buffer_m} m, overlap >= {min_share:.0%} of street length); "
          f"{n_tied:,} resolved a tie on the largest {fields[0]}")
    return out


def jenks_score(values, n_classes=JENKS_CLASSES):
    """Score 1..n on Jenks natural breaks; nulls score 1. Returns (scores, breaks)."""
    v = pd.to_numeric(pd.Series(values), errors="coerce")
    data = v.dropna().to_numpy(dtype=float)
    k = min(n_classes, len(np.unique(data)))
    breaks = jenkspy.jenks_breaks(data, n_classes=k)
    # [b_i, b_i+1) intervals, last one closed - same as classify_with_jenks
    scores = np.searchsorted(np.asarray(breaks[1:-1]), v.fillna(-np.inf).to_numpy(), side="right") + 1
    scores[v.isna().to_numpy()] = 1
    return pd.Series(scores, index=v.index, dtype="int16"), [float(b) for b in breaks]


def minmax(s):
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng else s * 0.0


def intersects_any(lines, polys):
    """Boolean per line: intersects at least one polygon."""
    pg = shapely.force_2d(polys.to_crs(lines.crs).geometry.values)
    bad = ~shapely.is_valid(pg)
    if bad.any():
        pg[bad] = shapely.make_valid(pg[bad])
    hit = np.unique(STRtree(pg).query(lines.geometry.values, predicate="intersects")[0])
    flag = np.zeros(len(lines), dtype="int16")
    flag[hit] = 1
    return flag

63,946 touching pairs


## 3. Hazard exposure

From `Hazard_Vulnerability.ipynb`. Flags are 1 where the street intersects any qualifying polygon.

| Field | Rule |
|---|---|
| `in_flood_zone` | any FEMA flood zone (100- or 500-year) |
| `in_flood_zone_100` | `FLOOD_YEAR = '100-year flood'` |
| `high_sev_fire` | `gridcode = 1` (probable high-severity fire) |
| `landslide_mean` / `landslide_max` | USGS landslide susceptibility over the raster cells each street crosses |

In [5]:
flood = pyogrio.read_dataframe(GDB, layer=FLOOD_LAYER, columns=["FLOOD_YEAR"])
fire = pyogrio.read_dataframe(GDB, layer=FIRE_LAYER, columns=["gridcode"], where="gridcode = 1")

streets["in_flood_zone"]     = intersects_any(streets, flood)
streets["in_flood_zone_100"] = intersects_any(streets, flood[flood.FLOOD_YEAR == "100-year flood"])
streets["high_sev_fire"]     = intersects_any(streets, fire)
streets[["in_flood_zone", "in_flood_zone_100", "high_sev_fire"]].sum()

C:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


in_flood_zone         696
in_flood_zone_100     396
high_sev_fire        1657
dtype: int64

**Landslide.** The raster is national and geographic (NAD83, ~0.0008 x 0.0014 degree cells), so each
street is projected to the raster CRS and sampled every quarter cell. Samples are de-duplicated to one per
(street, cell), which matches how zonal statistics counts each zone cell once. Every street gets at
least its midpoint sampled, so short streets that fall between cell centers still get a value.

In [6]:
with rasterio.open(LANDSLIDE_RASTER) as src:
    lines = streets.geometry.to_crs(src.crs).values
    step = min(abs(src.res[0]), abs(src.res[1])) / 4
    xmin, ymin, xmax, ymax = shapely.total_bounds(lines)
    pad = 2 * max(src.res)
    win = from_bounds(xmin - pad, ymin - pad, xmax + pad, ymax + pad, src.transform)
    win = win.round_offsets().round_lengths()
    grid = src.read(1, window=win, masked=True)
    wtf = src.window_transform(win)

lengths = shapely.length(lines)
n_pts = np.maximum(np.ceil(lengths / step).astype(int), 1) + 1
line_idx = np.repeat(np.arange(len(lines)), n_pts)
frac = np.concatenate([np.linspace(0, 1, n) for n in n_pts])
frac[np.r_[0, np.cumsum(n_pts)[:-1]]] = 0.5           # first sample of each line -> midpoint
pts = shapely.line_interpolate_point(lines[line_idx], frac, normalized=True)
r, c = rowcol(wtf, shapely.get_x(pts), shapely.get_y(pts))
r, c = np.asarray(r), np.asarray(c)
inside = (r >= 0) & (r < grid.shape[0]) & (c >= 0) & (c < grid.shape[1])

cells = pd.DataFrame({"line": line_idx[inside], "r": r[inside], "c": c[inside]}).drop_duplicates()
vals = grid[cells.r.to_numpy(), cells.c.to_numpy()]
cells["v"] = np.where(np.ma.getmaskarray(vals), np.nan, np.ma.getdata(vals)).astype(float)
stats = cells.dropna().groupby("line").v.agg(["mean", "max"])

streets["landslide_mean"] = stats["mean"].reindex(range(len(streets))).to_numpy()
streets["landslide_max"]  = stats["max"].reindex(range(len(streets))).to_numpy()
print(f"{len(pts):,} samples over {len(cells):,} street-cells; "
      f"{streets.landslide_mean.isna().sum()} streets with no data")
streets[["landslide_mean", "landslide_max"]].describe().round(2)

106,826 samples over 33,987 street-cells; 0 streets with no data


,landslide_mean,landslide_max
count,18285.00,18285.00
mean,14.06,17.04
std,20.74,24.02
min,0.00,0.00
25%,0.00,0.00
50%,2.17,3.00
75%,21.33,27.00
max,81.00,81.00


## 4. Equity

`assign_max_polygon_values` (Asset_Criticality) then `calculate_equity_index_feature_class`
(Asset_Scoring sec. 5):

* `<field>_max` - max of each ACS count over every block group the street intersects.
  **Fix:** streets that intersect none take the nearest block group (`equity_bg_match = 'nearest'`).
* `<field>_max_Normalized` - min-max 0-1 of each input.
* `Equity_Score` - equal-weight (0.25) sum of the four z-scores (sample std, as in `calculate_zscore`).
* `Equity_Score_Normalized` - min-max 0-1 of `Equity_Score`.

In [7]:
bg = pyogrio.read_dataframe(GDB, layer=BLOCK_GROUP_LAYER, columns=["GEOID"] + EQUITY_FIELDS)
bg = bg.to_crs(streets.crs)
bg["geometry"] = shapely.force_2d(bg.geometry.values)

hit = gpd.sjoin(streets[["geometry"]], bg, predicate="intersects", how="inner")
eq = hit.groupby(level=0)[EQUITY_FIELDS].max()
missing = streets.index.difference(eq.index)
near = gpd.sjoin_nearest(streets.loc[missing, ["geometry"]], bg, how="inner")
near = near[~near.index.duplicated()][EQUITY_FIELDS]
eq = pd.concat([eq, near]).reindex(streets.index)

EQ_MAX = [f"{f}_max" for f in EQUITY_FIELDS]
streets[EQ_MAX] = eq[EQUITY_FIELDS].astype(float).to_numpy()
streets["equity_bg_match"] = np.where(streets.index.isin(missing), "nearest", "intersect")
print(f"{len(missing)} streets intersect no block group -> nearest; "
      f"{streets[EQ_MAX].isna().any(axis=1).sum()} still null")

for f in EQ_MAX:
    streets[f"{f}_Normalized"] = minmax(streets[f])
z = (streets[EQ_MAX] - streets[EQ_MAX].mean()) / streets[EQ_MAX].std()
streets["Equity_Score"] = (z * 0.25).sum(axis=1)
streets["Equity_Score_Normalized"] = minmax(streets["Equity_Score"])
streets[EQ_MAX + ["Equity_Score", "Equity_Score_Normalized"]].describe().round(3)

32 streets intersect no block group -> nearest; 0 still null


,Below_Poverty_Household_max,Speak_English_Not_Well_Not_at_A_max,Vehicle_Available_0_max,With_Disability_max,Equity_Score,Equity_Score_Normalized
count,18285.000,18285.000,18285.000,18285.000,18285.000,18285.000
mean,24.088,20.319,9.392,31.103,0.000,0.145
std,27.116,36.912,18.927,38.038,0.771,0.162
min,0.000,0.000,0.000,0.000,-0.688,0.000
25%,0.000,0.000,0.000,0.000,-0.547,0.030
50%,15.000,0.000,0.000,19.000,-0.273,0.087
75%,39.000,26.000,10.000,43.000,0.213,0.190
max,142.000,196.000,92.000,257.000,4.060,1.000


## 5. Traffic (AADT)

StreetLight 2024 AADT joined with `best_overlap_join` (5 m buffer, >= 50% of the street covered), as
in `Asset_Criticality.ipynb`.

`Estimated_2024_AADT_Filled`:
1. the StreetLight value where the street matched (`aadt_source = 'streetlight'`);
2. **fix:** for unmatched trunk / primary / secondary / tertiary streets, the max AADT of touching
   streets, walked along unmatched chains (`'neighbor'`);
3. otherwise 0 (`'none'`), as before - local and service streets carry no modeled AADT.

`AADT_Jenks_Score` - 10-class Jenks on the filled value.

In [8]:
aadt_src = pyogrio.read_dataframe(GDB, layer=AADT_LAYER, columns=["Estimated_2024_AADT", "name"])
j = best_overlap_join(streets, aadt_src, ["Estimated_2024_AADT", "name"])
streets["Estimated_2024_AADT"] = pd.to_numeric(j["Estimated_2024_AADT"]).astype(float)
streets["aadt_zone_name"] = j["name"].astype("string")

raw = streets["Estimated_2024_AADT"]
major_gap = streets["class"].isin(MAJOR_CLASSES) & raw.isna()
nb, n_filled = neighbor_max_fill(raw, major_gap)
filled = raw.copy()
filled[major_gap] = nb[major_gap.to_numpy()]
streets["aadt_source"] = np.select([raw.notna(), filled.notna()], ["streetlight", "neighbor"], "none")
streets["Estimated_2024_AADT_Filled"] = filled.fillna(0).round().astype("int32")
print(f"major-road gaps: {major_gap.sum()}, filled from neighbors: {n_filled}")

streets["AADT_Jenks_Score"], aadt_breaks = jenks_score(streets["Estimated_2024_AADT_Filled"])
print("Jenks breaks:", [round(b) for b in aadt_breaks])
pd.crosstab(streets["class"], streets["aadt_source"])

  7,748 of 18,285 streets matched (buffer 5 m, overlap >= 50% of street length); 51 resolved a tie on the largest Estimated_2024_AADT
major-road gaps: 170, filled from neighbors: 168


Jenks breaks: [0, 553, 1918, 3914, 6091, 9025, 12406, 15446, 22554, 26727, 31624]


aadt_source,neighbor,none,streetlight
class,,,
living_street,0,11,0
primary,96,1,831
residential,0,1931,4375
secondary,4,0,297
service,0,8142,424
tertiary,20,1,815
trunk,48,0,394
unclassified,0,219,530
unknown,0,145,1


## 6. Detour length (single-link redundancy)

RA2CE single link redundancy rerun on the corrected network (`scripts/ra2ce_v2`:
`pipeline.py --stages slr`, then `export_slr.py`), keyed on OBJECTID like the OD table. For each link
RA2CE removes it and measures the shortest alternative between its endpoints:

* `slr_diff_length` - extra distance (m) of the detour, max over the segments the street covers;
* `slr_detour` - 1 when a detour exists, 0 when removing the link disconnects its endpoints;
* `slr_alt_length`, `slr_coverage_pct`, `slr_source_rows` - detail and QA.

`slr_diff_length_max` (name kept for the Criticality Index app):
1. the v2 value where a detour exists (`slr_source = 'detour'`);
2. 0 where there is no detour (`'no_detour'`), as in the published layer. Mostly dead-end service and
   residential streets. Scoring them 0 is a modeling choice: strictly, losing a link with no detour is
   the most disruptive case, but it is usually a driveway or cul-de-sac serving very few trips;
3. for trunk / primary / secondary / tertiary streets with no detour, the max of touching streets
   (`'neighbor'`). This is the `Asset_Criticality` neighbor fill, kept for the few major-road cases left.

`SLR_Diff_Length_Jenks_Score` - 10-class Jenks on the final value.

In [9]:
slr = pd.read_csv(SLR_UPDATE_CSV).set_index("OBJECTID")
assert set(slr.index) == set(streets.src_objectid), "SLR update OBJECTIDs differ from the base layer"
slr = slr.reindex(streets.src_objectid)

streets["slr_diff_length_raw"] = slr["slr_diff_length"].to_numpy()
streets["slr_alt_length"] = slr["slr_alt_length"].to_numpy()
streets["slr_detour_max"] = slr["slr_detour"].astype("Int16").to_numpy()
streets["slr_coverage_pct"] = slr["slr_coverage_pct"].to_numpy()
streets["slr_source_rows"] = slr["slr_source_rows"].astype("int16").to_numpy()

no_detour = (streets["slr_detour_max"] == 0).fillna(False).to_numpy(dtype=bool)
major = streets["class"].isin(MAJOR_CLASSES).to_numpy()
base = np.where(no_detour, 0.0, streets["slr_diff_length_raw"].to_numpy(dtype=float))
target = major & no_detour
nb, n_filled = neighbor_max_fill(base, target)

use_nb = target & ~np.isnan(nb)
final = base.copy()
final[use_nb] = nb[use_nb]
streets["slr_diff_length_max"] = final
streets["slr_source"] = np.select([use_nb, no_detour], ["neighbor", "no_detour"], "detour")
print(f"no detour: {no_detour.sum():,} streets, {target.sum()} on major roads, "
      f"{n_filled} of those filled from neighbors; NULL: {np.isnan(final).sum()}")

streets["SLR_Diff_Length_Jenks_Score"], slr_breaks = jenks_score(streets["slr_diff_length_max"])
print("Jenks breaks:", [round(b) for b in slr_breaks])
pd.crosstab(streets["class"], streets["slr_source"])

no detour: 6,704 streets, 53 on major roads, 53 of those filled from neighbors; NULL: 0


Jenks breaks: [0, 607, 2153, 7323, 16683, 39990, 62846, 71308, 82785, 99496, 123901]


slr_source,detour,neighbor,no_detour
class,,,
living_street,0,0,11
primary,911,17,0
residential,4483,0,1823
secondary,293,8,0
service,4067,0,4499
tertiary,810,26,0
trunk,440,2,0
unclassified,514,0,235
unknown,63,0,83


## 7. OD criticality

The OD update table selected by `OD_METHOD` (default `stochastic`: travel time by road class + stochastic
assignment, from `od_stochastic_assignment.ipynb`). Both methods produce the same 20 OD fields plus three
QA columns through the same street join (`export_protect_va.build_update`), keyed on the PROTECT_VA /
`Streets_Network_Tahoe` OBJECTID. Values are length-weighted means; `routed_*` and `par_alt_*` are
maxima. **Check `od_row_spread_pct` before quoting one figure for a street built from several
criticality rows.**

Under the stochastic method, traffic is an average over 100 routing draws, so a street can carry a
fraction of an origin's trips. `routed_* = 1` means the street carried flow in at least one draw.

The key is verified two ways: the OBJECTID sets must be identical, and, when the cached published layer
is present, `segment_id` and `length_m` must agree on every OBJECTID.

In [10]:
print(f"OD method: {OD_METHOD} ({OD_UPDATE_CSV.name})")
od = pd.read_csv(OD_UPDATE_CSV).set_index("OBJECTID").sort_index()
OD_FIELDS = [c for c in od.columns if c not in ("od_coverage_pct", "od_source_rows", "od_row_spread_pct")]
assert len(OD_FIELDS) == 20, OD_FIELDS
assert set(od.index) == set(streets.src_objectid), "OD update OBJECTIDs differ from the base layer"

if OD_PUBLISHED.exists():
    pub = pyogrio.read_dataframe(OD_PUBLISHED, columns=["OBJECTID", "segment_id", "length_m"],
                                 read_geometry=False).set_index("OBJECTID")
    chk = streets.set_index("src_objectid")[["segment_id", "length_m"]].join(pub, rsuffix="_pub")
    assert (chk.segment_id == chk.segment_id_pub).all(), "segment_id mismatch on OBJECTID"
    assert np.allclose(chk.length_m, chk.length_m_pub, atol=0.01), "length_m mismatch on OBJECTID"
    print("OBJECTID key verified against the published PROTECT_VA layer")

od = od.reindex(streets.src_objectid)
for c in od.columns:
    streets[c] = od[c].to_numpy()
for c in [c for c in OD_FIELDS if c.startswith(("routed_", "par_alt_"))] + ["od_source_rows"]:
    streets[c] = streets[c].astype("int16")
print(f"null OD cells: {int(streets[OD_FIELDS].isna().sum().sum())}")
streets[[c for c in OD_FIELDS if c.startswith("traffic_")]].gt(0).sum()

OD method: stochastic (protect_va_od_update_stochastic.csv)


OBJECTID key verified against the published PROTECT_VA layer
null OD cells: 0


traffic_evacuation             9629
traffic_law_enforcement        9568
traffic_medical_facilities     9874
traffic_emergency_shelters    10008
dtype: int64

## 8. Criticality score

Four sub-indices, built the same way as in `html/criticality-index.html`: each input is z-scored with the
population std (a null scores the mean, 0).

| Sub-index | Input | Weight (`CRIT_WEIGHTS`) |
|---|---|---|
| Traffic | `AADT_Jenks_Score` | 0.3 |
| Detour | `SLR_Diff_Length_Jenks_Score` | 0.3 |
| OD traffic | `OD_Traffic_Jenks_Score` | 0.3 |
| Equity | 0.25 x each of the four `*_max` counts | 0.1 |

**OD traffic** is `OD_Traffic_Avg`, the mean of the four OD `traffic_*` fields (population routed to basin
exits, law enforcement, medical facilities and emergency shelters), scored on 10 Jenks classes like AADT
and detour length. Jenks keeps a handful of very heavy corridors (Lake Tahoe Blvd, US 50) from flattening
every other street toward zero, as a raw z-score of a heavily skewed variable would. The utilitarian
`traffic_*` fields are used, not `prioritarian_*`, so equity is not counted twice.

`criticality_score` = weighted sum of the four sub-indices; `criticality_pctl` is its percentile rank
(0-100). **This no longer matches the app's default view** (three sub-indices at 1/3 each, no OD term);
the app recomputes from its own sliders.

In [11]:
assert abs(sum(CRIT_WEIGHTS.values()) - 1) < 1e-9, f"CRIT_WEIGHTS sum to {sum(CRIT_WEIGHTS.values())}"

def z_pop(s):
    s = pd.to_numeric(s, errors="coerce").astype(float)
    sd = s.std(ddof=0) or 1.0
    return ((s - s.mean()) / sd).fillna(0.0)

OD_TRAFFIC = [f"traffic_{k}" for k in ("evacuation", "law_enforcement", "medical_facilities", "emergency_shelters")]
streets["OD_Traffic_Avg"] = streets[OD_TRAFFIC].mean(axis=1)
streets["OD_Traffic_Jenks_Score"], od_breaks = jenks_score(streets["OD_Traffic_Avg"])
print("OD traffic Jenks breaks:", [round(b) for b in od_breaks])

streets["equity_subindex"]  = sum(0.25 * z_pop(streets[f]) for f in EQ_MAX)
streets["traffic_subindex"] = z_pop(streets["AADT_Jenks_Score"])
streets["detour_subindex"]  = z_pop(streets["SLR_Diff_Length_Jenks_Score"])
streets["od_subindex"]      = z_pop(streets["OD_Traffic_Jenks_Score"])
streets["criticality_score"] = (CRIT_WEIGHTS["traffic"] * streets["traffic_subindex"]
                                + CRIT_WEIGHTS["detour"] * streets["detour_subindex"]
                                + CRIT_WEIGHTS["od"] * streets["od_subindex"]
                                + CRIT_WEIGHTS["equity"] * streets["equity_subindex"])
streets["criticality_pctl"]  = (streets["criticality_score"].rank(pct=True) * 100).round(2)
streets[["equity_subindex", "traffic_subindex", "detour_subindex", "od_subindex",
         "criticality_score", "criticality_pctl"]].describe().round(3)

OD traffic Jenks breaks: [0, 135, 454, 866, 1344, 1971, 2981, 4499, 5827, 7002, 9619]


,equity_subindex,traffic_subindex,detour_subindex,od_subindex,criticality_score,criticality_pctl
count,18285.000,18285.000,18285.000,18285.000,18285.000,18285.000
mean,-0.000,-0.000,0.000,-0.000,-0.000,50.003
std,0.771,1.000,1.000,1.000,0.738,28.848
min,-0.688,-0.338,-0.303,-0.344,-0.365,5.370
25%,-0.547,-0.338,-0.303,-0.344,-0.341,24.820
50%,-0.273,-0.338,-0.303,-0.344,-0.283,49.260
75%,0.213,-0.338,-0.303,-0.344,-0.053,74.890
max,4.060,5.654,8.327,6.181,5.692,100.000


In [12]:
# Which streets the OD term moves most: top 15 by criticality, with their sub-indices
cols = ["name", "class", "OD_Traffic_Avg", "OD_Traffic_Jenks_Score", "AADT_Jenks_Score",
        "SLR_Diff_Length_Jenks_Score", "criticality_score", "criticality_pctl"]
streets.sort_values("criticality_score", ascending=False)[cols].head(15).round(2)

,name,class,OD_Traffic_Avg,OD_Traffic_Jenks_Score,AADT_Jenks_Score,SLR_Diff_Length_Jenks_Score,criticality_score,criticality_pctl
13219,Trooper Gary Gifford Highway,trunk,8173.06,10,9,9,5.69,100.00
9672,Trooper Gary Gifford Highway,trunk,8173.06,10,9,8,5.40,99.99
10975,Route 50,trunk,3571.52,7,6,10,4.64,99.99
10973,Lake Tahoe Boulevard,trunk,8066.01,10,10,5,4.64,99.98
11274,Route 50,trunk,3554.26,7,6,10,4.62,99.98
11364,Route 50,trunk,3554.26,7,6,10,4.56,99.97
15815,Route 50,trunk,3551.26,7,6,10,4.56,99.97
6770,Tahoe Boulevard,primary,3582.73,7,6,10,4.53,99.96
12205,Tahoe Boulevard,primary,3577.69,7,6,10,4.53,99.96
2362,Lake Tahoe Boulevard,trunk,8066.25,10,10,4,4.35,99.95


## 9. Schema and field dictionary

One table drives the output schema, the field order, the ArcGIS aliases and the field dictionary CSV.
`group` / `source` / `method` document where every field comes from.

In [13]:
OD_DESC = {
    "traffic": ("DOUBLE", "Population routed over the street"),
    "egalitarian": ("DOUBLE", "OD pairs routed over the street (unweighted)"),
    "prioritarian": ("DOUBLE", "Population routed, weighted by 1 + normalized poverty rate"),
    "routed": ("SHORT", "1 = carried routed flow"),
    "par_alt": ("SHORT", "1 = only on parallel alternative edges that never carry flow"),
}
OD_DEST = {"evacuation": "basin exits", "law_enforcement": "law enforcement",
           "medical_facilities": "medical facilities", "emergency_shelters": "emergency shelters"}
OD_RULE = {"evacuation": "nearest", "law_enforcement": "nearest",
           "medical_facilities": "gravity (beta 2) across all", "emergency_shelters": "gravity (beta 2) across all"}
OD_ROUTING = {"stochastic": "travel time by road class, 100-draw stochastic assignment",
              "ra2ce": "shortest distance (RA2CE)"}[OD_METHOD]
OD_SRC = OD_UPDATE_CSV.name

S = []  # (field, type, length, alias, group, source, method)
def f(name, typ, alias, group, source, method, length=None):
    S.append(dict(field=name, type=typ, length=length, alias=alias, group=group, source=source, method=method))

f("src_objectid", "LONG", "Source OBJECTID", "Identity", BASE_LAYER, "OBJECTID in Streets_Network_Tahoe and PROTECT_VA; join key")
f("segment_id", "TEXT", "Overture Segment ID", "Identity", BASE_LAYER, "Not unique: one Overture segment is split into several links", 64)
f("from_connector", "TEXT", "From Connector", "Identity", BASE_LAYER, "Overture connector ID", 64)
f("to_connector", "TEXT", "To Connector", "Identity", BASE_LAYER, "Overture connector ID", 64)
f("class", "TEXT", "Road Class", "Identity", BASE_LAYER, "Overture road class", 32)
f("name", "TEXT", "Street Name", "Identity", BASE_LAYER, "Overture primary name", 255)
f("length_m", "DOUBLE", "Length (m)", "Identity", BASE_LAYER, "Segment length")
f("DriveTime", "DOUBLE", "Drive Time (min)", "Identity", BASE_LAYER, "Free-flow drive time")
f("In_Basin", "SHORT", "In Basin", "Identity", BASE_LAYER, "1 = inside the Tahoe basin")

f("in_flood_zone", "SHORT", "In Flood Zone (100/500-yr)", "Hazard", FLOOD_LAYER, "1 = intersects any FEMA flood zone")
f("in_flood_zone_100", "SHORT", "In 100-yr Flood Zone", "Hazard", FLOOD_LAYER, "1 = intersects FLOOD_YEAR = '100-year flood'")
f("high_sev_fire", "SHORT", "High-Severity Fire", "Hazard", FIRE_LAYER, "1 = intersects gridcode = 1 (probable high severity)")
f("landslide_mean", "DOUBLE", "Landslide Susceptibility (mean)", "Hazard", LANDSLIDE_RASTER.name, "Mean over raster cells crossed")
f("landslide_max", "DOUBLE", "Landslide Susceptibility (max)", "Hazard", LANDSLIDE_RASTER.name, "Max over raster cells crossed")

for fld, lbl in zip(EQUITY_FIELDS, ["Households Below Poverty", "Limited English", "Zero-Vehicle Households", "Persons with Disability"]):
    f(f"{fld}_max", "DOUBLE", f"{lbl} (BG max)", "Equity", BLOCK_GROUP_LAYER, "Max over intersecting block groups (nearest if none)")
f("equity_bg_match", "TEXT", "Equity Block Group Match", "Equity", BLOCK_GROUP_LAYER, "intersect | nearest", 16)
for fld, lbl in zip(EQUITY_FIELDS, ["Households Below Poverty", "Limited English", "Zero-Vehicle Households", "Persons with Disability"]):
    f(f"{fld}_max_Normalized", "DOUBLE", f"{lbl} (0-1)", "Equity", "derived", "Min-max of the _max field")
f("Equity_Score", "DOUBLE", "Equity Score (z)", "Equity", "derived", "0.25 x z-score of each _max field")
f("Equity_Score_Normalized", "DOUBLE", "Equity Score (0-1)", "Equity", "derived", "Min-max of Equity_Score")

f("Estimated_2024_AADT", "DOUBLE", "StreetLight 2024 AADT", "Traffic", AADT_LAYER, "Best-overlap join, 5 m buffer, >= 50% of street")
f("aadt_zone_name", "TEXT", "StreetLight Zone", "Traffic", AADT_LAYER, "Name of the matched StreetLight zone", 255)
f("Estimated_2024_AADT_Filled", "LONG", "AADT (filled)", "Traffic", "derived", "StreetLight, else max of touching streets (major roads), else 0")
f("aadt_source", "TEXT", "AADT Source", "Traffic", "derived", "streetlight | neighbor | none", 16)
f("AADT_Jenks_Score", "SHORT", "AADT Jenks Score (1-10)", "Traffic", "derived", "10-class Jenks on AADT (filled)")

SLR_SRC = "ra2ce_v2 protect_va_slr_update.csv"
f("slr_diff_length_raw", "DOUBLE", "SLR Detour Length, v2 (m)", "Detour", SLR_SRC, "Max extra detour distance over covered segments; NULL = no detour")
f("slr_alt_length", "DOUBLE", "SLR Alternative Route Length (m)", "Detour", SLR_SRC, "Alternative route for the segment setting slr_diff_length_raw")
f("slr_detour_max", "SHORT", "SLR Detour Exists", "Detour", SLR_SRC, "Min over covered segments: 0 = some segment has no alternative route")
f("slr_coverage_pct", "DOUBLE", "SLR Coverage (%)", "Detour", SLR_SRC, "Share of the street covered by SLR segments")
f("slr_source_rows", "SHORT", "SLR Source Rows", "Detour", SLR_SRC, "SLR segments contributing to the street")
f("slr_diff_length_max", "DOUBLE", "SLR Detour Length (m)", "Detour", "derived", "v2 value; 0 where no detour; neighbor max on major roads with no detour")
f("slr_source", "TEXT", "SLR Source", "Detour", "derived", "detour | no_detour | neighbor", 16)
f("SLR_Diff_Length_Jenks_Score", "SHORT", "SLR Detour Jenks Score (1-10)", "Detour", "derived", "10-class Jenks on slr_diff_length_max; null = 1")

for dest_key, dest in OD_DEST.items():
    for metric, (typ, desc) in OD_DESC.items():
        f(f"{metric}_{dest_key}", typ, f"{metric.replace('_', ' ').title()} - {dest.title()}", "OD criticality",
          OD_SRC, f"{desc}; {OD_RULE[dest_key]} {dest}; {OD_ROUTING}")
f("od_coverage_pct", "DOUBLE", "OD Coverage (%)", "OD criticality", OD_SRC, "Share of the street covered by criticality rows")
f("od_source_rows", "SHORT", "OD Source Rows", "OD criticality", OD_SRC, "Criticality rows contributing to the street")
f("od_row_spread_pct", "DOUBLE", "OD Row Spread (%)", "OD criticality", OD_SRC, "Disagreement among contributing rows; check before quoting")

f("OD_Traffic_Avg", "DOUBLE", "OD Traffic, Mean of 4 Analyses", "Criticality", "derived", "Mean of traffic_evacuation / _law_enforcement / _medical_facilities / _emergency_shelters")
f("OD_Traffic_Jenks_Score", "SHORT", "OD Traffic Jenks Score (1-10)", "Criticality", "derived", "10-class Jenks on OD_Traffic_Avg")
f("equity_subindex", "DOUBLE", "Equity Sub-Index", "Criticality", "derived", "0.25 x population z of each _max field")
f("traffic_subindex", "DOUBLE", "Traffic Sub-Index", "Criticality", "derived", "Population z of AADT_Jenks_Score")
f("detour_subindex", "DOUBLE", "Detour Sub-Index", "Criticality", "derived", "Population z of SLR_Diff_Length_Jenks_Score")
f("od_subindex", "DOUBLE", "OD Traffic Sub-Index", "Criticality", "derived", "Population z of OD_Traffic_Jenks_Score")
f("criticality_score", "DOUBLE", "Criticality Score", "Criticality", "derived",
  "Weighted sum: " + ", ".join(f"{k} {v:g}" for k, v in CRIT_WEIGHTS.items()))
f("criticality_pctl", "DOUBLE", "Criticality Percentile", "Criticality", "derived", "Percentile rank of criticality_score (0-100)")

schema = pd.DataFrame(S)
missing_cols = set(schema.field) - set(streets.columns)
extra_cols = set(streets.columns) - set(schema.field) - {"geometry"}
assert not missing_cols, f"schema fields not built: {missing_cols}"
assert not extra_cols, f"built fields missing from schema: {extra_cols}"

clean = streets[schema.field.tolist() + ["geometry"]].copy()
schema["nulls"] = schema.field.map(clean.isna().sum())
schema.to_csv(OUT_DIR / "streets_network_tahoe_clean_fields.csv", index=False)
print(f"{len(schema)} fields")
schema[["field", "type", "group", "nulls", "method"]]

69 fields


,field,type,group,nulls,method
0,src_objectid,LONG,Identity,0,OBJECTID in Streets_Network_Tahoe and PROTECT_...
1,segment_id,TEXT,Identity,0,Not unique: one Overture segment is split into...
2,from_connector,TEXT,Identity,0,Overture connector ID
3,to_connector,TEXT,Identity,0,Overture connector ID
4,class,TEXT,Identity,0,Overture road class
5,name,TEXT,Identity,0,Overture primary name
6,length_m,DOUBLE,Identity,0,Segment length
7,DriveTime,DOUBLE,Identity,0,Free-flow drive time
8,In_Basin,SHORT,Identity,0,1 = inside the Tahoe basin
9,in_flood_zone,SHORT,Hazard,0,1 = intersects any FEMA flood zone


## 10. QA against the published layer

Every field present in both layers, compared on `src_objectid`. Changes are expected where a fix applies;
anything else is worth a look before republishing.

In [14]:
qa_rows = []
cl = clean.set_index("src_objectid")
for fld in [c for c in schema.field if c in published.columns]:
    a, b = published[fld], cl[fld]
    num = pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b)
    if num:
        a, b = a.astype(float), b.astype(float)
        same = (np.isclose(a, b, rtol=1e-6, atol=1e-6) | (a.isna() & b.isna()))
    else:
        same = (a.astype("string") == b.astype("string")).fillna(False) | (a.isna() & b.isna())
    qa_rows.append(dict(field=fld, nulls_before=int(a.isna().sum()), nulls_after=int(b.isna().sum()),
                        changed=int((~same).sum()),
                        corr=round(a.corr(b), 4) if num and a.notna().sum() > 1 else None))
qa = pd.DataFrame(qa_rows)
qa.to_csv(OUT_DIR / "streets_network_tahoe_clean_qa.csv", index=False)

dropped = [c for c in published.columns if c not in schema.field.tolist()]
print("Dropped from the published layer:", dropped)
print("New fields:", [c for c in schema.field if c not in published.columns])
qa

Dropped from the published layer: ['surface', 'speed_limit_kmh', 'created_user', 'created_date', 'last_edited_user', 'last_edited_date', 'diff_length_max', 'detour_max', 'Shape_Length', 'slr_diff_length_max_from_neighbors']
New fields: ['src_objectid', 'in_flood_zone_100', 'equity_bg_match', 'aadt_source', 'slr_diff_length_raw', 'slr_alt_length', 'slr_coverage_pct', 'slr_source_rows', 'slr_source', 'od_coverage_pct', 'od_source_rows', 'od_row_spread_pct', 'OD_Traffic_Avg', 'OD_Traffic_Jenks_Score', 'equity_subindex', 'traffic_subindex', 'detour_subindex', 'od_subindex', 'criticality_score', 'criticality_pctl']


C:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]


,field,nulls_before,nulls_after,changed,corr
0,segment_id,0,0,0,NaN
1,from_connector,0,0,0,NaN
2,to_connector,0,0,0,NaN
3,class,0,0,0,NaN
4,name,0,0,0,NaN
5,length_m,0,0,0,1.0000
6,DriveTime,0,0,0,1.0000
7,In_Basin,0,0,0,NaN
8,in_flood_zone,0,0,0,1.0000
9,high_sev_fire,0,0,0,1.0000


In [15]:
checks = {
    "row count matches the base layer": len(clean) == len(published),
    "src_objectid unique": clean.src_objectid.is_unique,
    "no NULL OD fields": int(clean[OD_FIELDS].isna().sum().sum()) == 0,
    "no NULL equity inputs": int(clean[EQ_MAX].isna().sum().sum()) == 0,
    "no NULL hazard flags": int(clean[["in_flood_zone", "in_flood_zone_100", "high_sev_fire"]].isna().sum().sum()) == 0,
    "no NULL AADT (filled)": int(clean.Estimated_2024_AADT_Filled.isna().sum()) == 0,
    "no NULL or negative SLR detour length": bool(clean.slr_diff_length_max.notna().all()
                                                  and (clean.slr_diff_length_max >= 0).all()),
    "Jenks scores within 1..10": bool(clean[["AADT_Jenks_Score", "SLR_Diff_Length_Jenks_Score",
                                             "OD_Traffic_Jenks_Score"]].isin(range(1, 11)).all().all()),
    "no NULL criticality score": int(clean.criticality_score.isna().sum()) == 0,
}
for k, v in checks.items():
    print(("PASS " if v else "FAIL ") + k)
assert all(checks.values()), "QA check failed"

PASS row count matches the base layer
PASS src_objectid unique
PASS no NULL OD fields
PASS no NULL equity inputs
PASS no NULL hazard flags
PASS no NULL AADT (filled)
PASS no NULL or negative SLR detour length
PASS Jenks scores within 1..10
PASS no NULL criticality score


## 11. Write outputs

* `scripts/outputs/streets_network_clean/streets_network_tahoe_clean.gpkg` (gitignored), always written;
* `Streets_Network_Tahoe_Clean` in `PROTECT_analysis.gdb`, built with an explicit schema (types, text
  lengths, aliases) from the table above. It is built in a local FGDB next to the GeoPackage and then
  copied to F: in one operation. Rows are inserted in `src_objectid` order, so the new OBJECTIDs match
  the source ones.

In [16]:
clean.to_file(OUT_GPKG, layer="streets_network_tahoe_clean", driver="GPKG")
print(f"wrote {OUT_GPKG}")

wrote C:\Users\amcclary\Documents\GitHub\PROTECT\scripts\outputs\streets_network_clean\streets_network_tahoe_clean.gpkg


In [17]:
if WRITE_FGDB:
    import arcpy

    # Build in a local FGDB, then copy to F: in one operation. Schema calls against the network
    # GDB take ~30 s each, so adding 63 fields there one at a time stalls for half an hour.
    out_path = str(GDB / OUT_FC)
    if arcpy.Exists(out_path) and not OVERWRITE:
        raise RuntimeError(f"{OUT_FC} exists; set OVERWRITE = True to replace it")

    # The local FGDB is scratch, so it is rebuilt from nothing on every run. Deleting just the
    # feature class inside it has left the FGDB catalog still listing it (Exists misreports and
    # CreateFeatureclass then fails), so the whole folder goes.
    import shutil
    local_gdb = OUT_DIR / "streets_network_tahoe_clean.gdb"
    arcpy.ClearWorkspaceCache_management()
    if local_gdb.exists():
        shutil.rmtree(local_gdb)
    arcpy.management.CreateFileGDB(str(OUT_DIR), local_gdb.name)
    local_fc = str(local_gdb / OUT_FC)

    sr = arcpy.SpatialReference(clean.crs.to_epsg())
    arcpy.management.CreateFeatureclass(str(local_gdb), OUT_FC, "POLYLINE", spatial_reference=sr)
    arcpy.management.AddFields(local_fc, [
        [r.field, r.type, r.alias, int(r.length) if pd.notna(r.length) else None]
        for r in schema.itertuples()])

    fields = schema.field.tolist()
    text_fields = set(schema.loc[schema.type == "TEXT", "field"])
    int_fields = set(schema.loc[schema.type.isin(["SHORT", "LONG"]), "field"])

    def cell(fld, v):
        if pd.isna(v):
            return None
        if fld in text_fields:
            return str(v)
        return int(v) if fld in int_fields else float(v)

    wkbs = shapely.to_wkb(clean.geometry.values)
    t0 = time.time()
    with arcpy.da.InsertCursor(local_fc, ["SHAPE@"] + fields) as cur:
        for wkb, rec in zip(wkbs, clean[fields].itertuples(index=False, name=None)):
            cur.insertRow([arcpy.FromWKB(wkb, sr)] + [cell(f, v) for f, v in zip(fields, rec)])

    n = int(arcpy.management.GetCount(local_fc)[0])
    off = sum(1 for oid, src in arcpy.da.SearchCursor(local_fc, ["OID@", "src_objectid"]) if oid != src)
    print(f"built {n:,} features in {local_fc} in {time.time() - t0:.0f}s; "
          f"{off} OBJECTIDs differ from src_objectid")

    t0 = time.time()
    if arcpy.Exists(out_path):
        arcpy.management.Delete(out_path)
    arcpy.management.Copy(local_fc, out_path)
    print(f"copied to {out_path} in {time.time() - t0:.0f}s")

built 18,285 features in C:\Users\amcclary\Documents\GitHub\PROTECT\scripts\outputs\streets_network_clean\streets_network_tahoe_clean.gdb\Streets_Network_Tahoe_Clean in 54s; 0 OBJECTIDs differ from src_objectid


copied to F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\PROTECT_analysis.gdb\Streets_Network_Tahoe_Clean in 108s
